# 01. Data Cleaning & Preprocessing (数据清洗)

**Goal:** Transform raw TMDB data into a clean format suitable for analysis.

**Steps:**
1. Load Data
2. Filter Invalid Rows (Revenue/Budget = 0)
3. Parse JSON Columns (Genres, Production Companies, etc.)
4. Feature Engineering (Date extraction)
5. Save Processed Data

In [8]:
import pandas as pd
import numpy as np
import ast  # For safe evaluation of stringified lists
import warnings
import os

warnings.filterwarnings('ignore')

# Set file paths relative to this notebook
# Assuming notebook is in 'notebooks/' folder
RAW_DATA_PATH = '../data/raw/TMDB_movie_dataset_v11.csv'
PROCESSED_DATA_PATH = '../data/processed/clean_tmdb_movies.csv'

# Check if file exists
if not os.path.exists(RAW_DATA_PATH):
    print(f"Error: File not found at {RAW_DATA_PATH}")
    print("Current working directory:", os.getcwd())

In [9]:
# 1. Load Data
try:
    df = pd.read_csv(RAW_DATA_PATH)
    print(f"Original Shape: {df.shape}")
    display(df.head(2))
except Exception as e:
    print(f"Failed to load data: {e}")

Original Shape: (1379467, 24)


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."


In [10]:
# 2. Filter out movies with 0 revenue or budget
# Many older or small movies have 0 revenue recorded. For serious box office analysis, these are noise.
# We will filter keeping only movies with revenue > 10000 and budget > 10000 to be safe.

print("Rows before filtering:", len(df))
clean_df = df[ (df['revenue'] > 10000) & (df['budget'] > 10000) ].copy()
print(f"Rows after filtering (Revenue > 10k & Budget > 10k): {len(clean_df)}")

Rows before filtering: 1379467
Rows after filtering (Revenue > 10k & Budget > 10k): 10459


In [11]:
# 3. Parse Metadata Columns
# Based on inspection, these columns are simple comma-separated strings.
# Example: "Action, Science Fiction, Adventure"

def parse_list_col(x):
    if isinstance(x, str):
        # Split by comma and strip whitespace
        return [item.strip() for item in x.split(',') if item.strip()]
    return []

# Columns that need parsing
list_cols = ['genres', 'production_companies', 'production_countries', 'spoken_languages']

# Inspect before
print("Sample genre before parsing:", clean_df['genres'].iloc[0])

for col in list_cols:
    # Check if column exists to avoid errors
    if col in clean_df.columns:
        print(f"Parsing {col}...")
        clean_df[col] = clean_df[col].apply(parse_list_col)

# Check the results
print("Sample genre after parsing:", clean_df['genres'].iloc[0])
clean_df[list_cols].head()

Sample genre before parsing: Action, Science Fiction, Adventure
Parsing genres...
Parsing production_companies...
Parsing production_countries...
Parsing spoken_languages...
Sample genre after parsing: ['Action', 'Science Fiction', 'Adventure']


,genres,production_companies,production_countries,spoken_languages
0,"[Action, Science Fiction, Adventure]","[Legendary Pictures, Syncopy, Warner Bros. Pic...","[United Kingdom, United States of America]","[English, French, Japanese, Swahili]"
1,"[Adventure, Drama, Science Fiction]","[Legendary Pictures, Syncopy, Lynda Obst Produ...","[United Kingdom, United States of America]",[English]
2,"[Drama, Action, Crime, Thriller]","[DC Comics, Legendary Pictures, Syncopy, Isobe...","[United Kingdom, United States of America]","[English, Mandarin]"
3,"[Action, Adventure, Fantasy, Science Fiction]","[Dune Entertainment, Lightstorm Entertainment,...","[United States of America, United Kingdom]","[English, Spanish]"
4,"[Science Fiction, Action, Adventure]",[Marvel Studios],[United States of America],"[English, Hindi, Russian]"


In [12]:
# 4. Feature Engineering: Date Features
# Convert release_date to datetime
clean_df['release_date'] = pd.to_datetime(clean_df['release_date'], errors='coerce')

# Extract Year and Month
clean_df['release_year'] = clean_df['release_date'].dt.year
clean_df['release_month'] = clean_df['release_date'].dt.month

# Drop rows with missing dates
clean_df = clean_df.dropna(subset=['release_date'])

# Calculate ROI (Return on Investment) already here?
# ROI = (Revenue - Budget) / Budget
clean_df['roi'] = (clean_df['revenue'] - clean_df['budget']) / clean_df['budget']

print("Date columns processed.")

Date columns processed.


In [13]:
# 5. Save to Processed
# Since list columns can be tricky in CSVs (they load as strings), we accept that they will be saved as string reps.

if not os.path.exists('../data/processed'):
    os.makedirs('../data/processed')

clean_df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Saved processed data to {PROCESSED_DATA_PATH}")
print(f"Final Clean Shape: {clean_df.shape}")

Saved processed data to ../data/processed/clean_tmdb_movies.csv
Final Clean Shape: (10000, 27)
